## Feature Importance — Built-in (XGBoost)

In [1]:
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

xgb_balanced = joblib.load("xgb_balanced.joblib")

df = df = pd.read_csv("cleaned_data.csv", index_col="id")

x = df.drop(columns=["default_payment_next_month"])
y = df["default_payment_next_month"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [2]:
import pandas as pd

xgb_final_model = xgb_balanced

feature_names = xgb_final_model.named_steps["preprocessor"].get_feature_names_out()

feature_importance = xgb_final_model.named_steps["classifier"].feature_importances_

fi = pd.Series(feature_importance, feature_names).sort_values(ascending=False)

fi

remainder__pay_1        0.354756
remainder__pay_2        0.141171
remainder__pay_3        0.075180
remainder__pay_6        0.042797
remainder__pay_4        0.040524
remainder__pay_5        0.035028
num__limit_bal          0.026344
num__pay_amt2           0.025091
num__pay_amt4           0.021925
num__pay_amt1           0.021735
num__pay_amt3           0.021623
num__bill_amt2          0.021070
num__bill_amt1          0.020943
remainder__education    0.017443
num__pay_amt6           0.016607
num__pay_amt5           0.015829
remainder__sex          0.015474
num__bill_amt4          0.015037
remainder__marriage     0.014966
num__bill_amt3          0.014459
num__bill_amt6          0.014359
num__bill_amt5          0.014336
num__age                0.013299
dtype: float32

### What this importance actually measures?
- Feature importance in XGBoost quantifies how much each input feature contributes to the model's prediction. helping interpret model and guide feature selection.

### Why this can be misleading?
- BEcause it depends on model if model overfits then its feature importance can not be trusted, Also feature importance is biased toward high cardinality or continuous feature which provide more opportunities to split the data but may not be able to create meaningful split.

### Why correlated features distort this view?
- when features are correlated models can use them interchangeably which can cause importance to be spread across group of correlated features.
- Removing one correlated feature may not degrade performance because other can "pick-up" the signal this affect importance of uncorrelated feature as they might be in top rankings of importance due to some feature which is totally redundant and can be removed. 

## Permutation Importance — Model-Agnostic

In [3]:
from sklearn.inspection import permutation_importance

r = permutation_importance(
    xgb_final_model,
    x_test,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring='roc_auc'
)
prem_importance = pd.Series(r.importances_mean, index=feature_names).sort_values(ascending=False)

prem_importance

num__bill_amt4          0.072233
num__limit_bal          0.022113
num__pay_amt4           0.014249
remainder__pay_1        0.007324
remainder__pay_2        0.007141
num__pay_amt5           0.006526
num__pay_amt6           0.005928
remainder__pay_3        0.004406
num__pay_amt3           0.003810
num__bill_amt5          0.003761
remainder__pay_4        0.003232
num__bill_amt1          0.003103
num__bill_amt6          0.002663
num__pay_amt2           0.002605
num__pay_amt1           0.002461
remainder__marriage     0.002161
num__bill_amt2          0.001913
remainder__education    0.001492
remainder__pay_6        0.001457
remainder__sex          0.001209
num__age                0.001123
num__bill_amt3          0.000352
remainder__pay_5        0.000160
dtype: float64

In [4]:
(prem_importance - fi).sort_values(ascending=True)

remainder__pay_1       -0.347432
remainder__pay_2       -0.134030
remainder__pay_3       -0.070774
remainder__pay_6       -0.041340
remainder__pay_4       -0.037292
remainder__pay_5       -0.034869
num__pay_amt2          -0.022486
num__pay_amt1          -0.019275
num__bill_amt2         -0.019157
num__bill_amt1         -0.017840
num__pay_amt3          -0.017813
remainder__education   -0.015951
remainder__sex         -0.014265
num__bill_amt3         -0.014107
remainder__marriage    -0.012806
num__age               -0.012176
num__bill_amt6         -0.011696
num__pay_amt6          -0.010679
num__bill_amt5         -0.010575
num__pay_amt5          -0.009303
num__pay_amt4          -0.007676
num__limit_bal         -0.004231
num__bill_amt4          0.057196
dtype: float64

- Permutation importance is calculated by shuffling the values of a single feature in the evaluation data (while keeping the trained model fixed) and measuring how much the model’s performance degrades. This process is repeated multiple times and averaged. Features whose shuffling causes a larger drop in performance are considered more important.

### Which features stay important?
- `bill_amt4`, `pay_amt4` and `limit_bal` stay important while others losses their importance.

### Which drop?
- `pay_1`, `pay_2` and `pay_3` looses their importance.

### What does this say about redundancy?
- If permuting a feature does not significantly hurts the performance, it suggests the feature's importance is largely duplicated by others and feature is redundant rather than uniquely important.